# Calibración de cámaras

Calibraremos una cámara a partir de varias vistas de un tablero de ajedrez y usaremos los parámetros estimados para corregir distorsión y proyectar un cubo 3D.

**Conceptos:** parámetros intrínsecos, distorsión radial/tangencial, parámetros extrínsecos, error de reproyección y patrones ChArUco.

**Resultado esperado:** una matriz intrínseca de 3 × 3, coeficientes de distorsión, errores de reproyección interpretables e imágenes corregidas. Los valores exactos dependen de las vistas válidas.


## 1. Dependencias y datos

La calibración requiere varias vistas del mismo patrón, con cambios de orientación y posición. La siguiente descarga es idempotente: solo obtiene las imágenes faltantes.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import cv2
import matplotlib.pyplot as plt
import numpy as np

image_dir = Path("camera_calibration_images")
image_dir.mkdir(exist_ok=True)

base_url = (
    "https://github.com/YoniChechik/AI_is_Math/raw/master/"
    "c_07_camera_calibration/images"
)
for image_number in range(1, 18):
    destination = image_dir / f"{image_number}.jpeg"
    if not destination.exists():
        urlretrieve(f"{base_url}/{image_number}.jpeg", destination)

image_paths = sorted(
    image_dir.glob("*.jpeg"),
    key=lambda path: int(path.stem),
)
if not image_paths:
    raise FileNotFoundError("No se encontraron imágenes de calibración.")

print(f"Se encontraron {len(image_paths)} imágenes.")
print("OpenCV:", cv2.__version__)


Mostramos dos vistas para comprobar que el tablero aparece en distintas poses. Esta variación es necesaria para estimar de manera estable la geometría de la cámara.

**Resultado esperado:** dos imágenes del mismo tablero con perspectiva y distorsión diferentes.


In [ ]:
sample_indices = [0, min(11, len(image_paths) - 1)]
figure, axes = plt.subplots(1, 2, figsize=(14, 5))

for axis, sample_index in zip(axes, sample_indices):
    image_bgr = cv2.imread(str(image_paths[sample_index]))
    if image_bgr is None:
        raise ValueError(f"No se pudo leer {image_paths[sample_index]}.")
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    axis.imshow(image_rgb)
    axis.set_title(image_paths[sample_index].name)
    axis.axis("off")

plt.tight_layout()
plt.show()


## 2. Coordenadas conocidas del tablero

pattern_size contiene las esquinas **internas**: 9 columnas y 6 filas. Como el tablero es plano, todos sus puntos tienen Z = 0. El tamaño real del cuadrado fija la unidad de las traslaciones estimadas; aquí usamos 2.88 cm.


In [ ]:
pattern_size = (9, 6)
square_size_cm = 2.88

# Cada fila representa un punto 3D conocido del tablero: (X, Y, Z).
board_points_3d = np.zeros(
    (pattern_size[0] * pattern_size[1], 3),
    dtype=np.float32,
)
board_points_3d[:, :2] = np.mgrid[
    0 : pattern_size[0],
    0 : pattern_size[1],
].T.reshape(-1, 2)
board_points_3d *= square_size_cm

print("Forma de los puntos 3D:", board_points_3d.shape)
print("Primeros cinco puntos, en cm:")
print(board_points_3d[:5])


## 3. Detección de esquinas

findChessboardCorners localiza el patrón y cornerSubPix refina cada posición a precisión subpíxel. Guardamos juntos la imagen, los puntos 2D y los puntos 3D; así evitamos desalinear poses si alguna vista falla.

**Resultado esperado:** varias vistas marcadas con esquinas de colores y un resumen del número de detecciones exitosas.


In [ ]:
object_points_per_view = []
image_points_per_view = []
valid_views = []
image_size = None

termination_criteria = (
    cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
    30,
    0.001,
)

figure, axes = plt.subplots(3, 4, figsize=(16, 11))
axes = axes.ravel()
displayed_views = 0

for image_path in image_paths:
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        print(f"Advertencia: no se pudo leer {image_path.name}")
        continue

    gray_image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    current_size = (gray_image.shape[1], gray_image.shape[0])
    if image_size is None:
        image_size = current_size
    elif current_size != image_size:
        raise ValueError("Todas las imágenes deben tener la misma resolución.")

    found, corners = cv2.findChessboardCorners(
        gray_image,
        pattern_size,
    )
    if not found:
        print(f"Sin patrón completo: {image_path.name}")
        continue

    refined_corners = cv2.cornerSubPix(
        gray_image,
        corners,
        winSize=(11, 11),
        zeroZone=(-1, -1),
        criteria=termination_criteria,
    )

    object_points_per_view.append(board_points_3d.copy())
    image_points_per_view.append(refined_corners)
    valid_views.append(
        {
            "path": image_path,
            "image_bgr": image_bgr,
            "corners": refined_corners,
        }
    )

    if displayed_views < len(axes):
        annotated = image_bgr.copy()
        cv2.drawChessboardCorners(
            annotated,
            pattern_size,
            refined_corners,
            True,
        )
        axes[displayed_views].imshow(
            cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        )
        axes[displayed_views].set_title(image_path.name)
        axes[displayed_views].axis("off")
        displayed_views += 1

for axis in axes[displayed_views:]:
    axis.axis("off")
plt.tight_layout()
plt.show()

if len(valid_views) < 3:
    raise RuntimeError(
        "Se requieren al menos tres vistas válidas; idealmente se usan más."
    )

print(f"Detecciones válidas: {len(valid_views)}/{len(image_paths)}")


## 4. Estimación de parámetros

La matriz intrínseca K contiene las distancias focales fx, fy y el punto principal (cx, cy). Los coeficientes modelan distorsión radial y tangencial. Para cada vista también obtenemos una rotación y una traslación.

El RMS entregado por OpenCV resume error de reproyección en píxeles: proyectamos los puntos 3D con los parámetros estimados y los comparamos con las esquinas observadas. Menor es mejor, aunque su interpretación depende de la resolución y calidad de las imágenes.


In [ ]:
(
    calibration_rms,
    camera_matrix,
    distortion_coefficients,
    rotation_vectors,
    translation_vectors,
) = cv2.calibrateCamera(
    object_points_per_view,
    image_points_per_view,
    image_size,
    None,
    None,
)

print(f"RMS global de reproyección: {calibration_rms:.4f} píxeles")
print("\nMatriz intrínseca K:")
print(np.round(camera_matrix, 3))
print("\nCoeficientes de distorsión:")
print(np.round(distortion_coefficients.ravel(), 5))


Calculamos además el error medio por vista. Esto permite detectar una fotografía borrosa o una detección de esquinas menos precisa que el RMS global podría ocultar.

**Resultado esperado:** errores del mismo orden entre vistas. Una vista claramente peor merece inspección.


In [ ]:
reprojection_errors = []

for view_index, (points_3d, observed_points_2d) in enumerate(
    zip(object_points_per_view, image_points_per_view)
):
    projected_points_2d, _ = cv2.projectPoints(
        points_3d,
        rotation_vectors[view_index],
        translation_vectors[view_index],
        camera_matrix,
        distortion_coefficients,
    )
    error = cv2.norm(
        observed_points_2d,
        projected_points_2d,
        cv2.NORM_L2,
    ) / len(projected_points_2d)
    reprojection_errors.append(error)

for view, error in zip(valid_views, reprojection_errors):
    print(f"{view['path'].name}: {error:.4f} píxeles")

print(
    f"\nError medio por vista: {np.mean(reprojection_errors):.4f} píxeles"
)


## 5. Corrección de distorsión

undistort remapea los píxeles usando K y los coeficientes estimados. Compare especialmente líneas rectas cercanas a los bordes.

**Resultado esperado:** las líneas del tablero deberían verse más rectas; pueden aparecer bordes negros porque algunos píxeles corregidos quedan fuera de la imagen original.


In [ ]:
sample_view = valid_views[0]["image_bgr"]
corrected_view = cv2.undistort(
    sample_view,
    camera_matrix,
    distortion_coefficients,
)

figure, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(sample_view, cv2.COLOR_BGR2RGB))
axes[0].set_title("Imagen original")
axes[1].imshow(cv2.cvtColor(corrected_view, cv2.COLOR_BGR2RGB))
axes[1].set_title("Imagen sin distorsión")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()


## 6. Proyección de un cubo 3D

Los parámetros extrínsecos ubican el tablero respecto de la cámara. Definimos un cubo apoyado en el tablero y proyectamos sus ocho vértices con projectPoints.

**Resultado esperado:** el cubo debe permanecer anclado al patrón y respetar su perspectiva.


In [ ]:
cube_size = 3 * square_size_cm
cube_points_3d = cube_size * np.float32(
    [
        [0, 0, 0],
        [0, 1, 0],
        [1, 1, 0],
        [1, 0, 0],
        [0, 0, -1],
        [0, 1, -1],
        [1, 1, -1],
        [1, 0, -1],
    ]
)

def draw_cube(image_bgr, projected_points):
    # Dibuja base, pilares y cara superior de un cubo proyectado.
    points = np.round(projected_points).astype(int).reshape(-1, 2)
    result = image_bgr.copy()

    result = cv2.drawContours(
        result,
        [points[:4]],
        contourIdx=-1,
        color=(0, 180, 0),
        thickness=-1,
    )
    for bottom_index, top_index in zip(range(4), range(4, 8)):
        result = cv2.line(
            result,
            tuple(points[bottom_index]),
            tuple(points[top_index]),
            color=(0, 0, 255),
            thickness=3,
        )
    result = cv2.drawContours(
        result,
        [points[4:]],
        contourIdx=-1,
        color=(255, 0, 0),
        thickness=3,
    )
    return result

figure, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for view_index, axis in enumerate(axes):
    if view_index >= len(valid_views):
        axis.axis("off")
        continue

    projected_cube, _ = cv2.projectPoints(
        cube_points_3d,
        rotation_vectors[view_index],
        translation_vectors[view_index],
        camera_matrix,
        distortion_coefficients,
    )
    cube_view = draw_cube(
        valid_views[view_index]["image_bgr"],
        projected_cube,
    )
    axis.imshow(cv2.cvtColor(cube_view, cv2.COLOR_BGR2RGB))
    axis.set_title(valid_views[view_index]["path"].name)
    axis.axis("off")

plt.tight_layout()
plt.show()


## 7. Introducción a ChArUco

Un tablero ChArUco combina esquinas de ajedrez precisas con identificadores ArUco. Los identificadores permiten recuperar correspondencias incluso cuando el tablero está parcialmente visible.

En este ejemplo generamos un tablero y verificamos que OpenCV detecta sus esquinas. Se requiere una instalación de OpenCV que incluya el módulo aruco.


In [ ]:
if not hasattr(cv2, "aruco"):
    raise ImportError(
        "Este ejemplo requiere opencv-contrib-python, que incluye cv2.aruco."
    )

aruco_dictionary = cv2.aruco.getPredefinedDictionary(
    cv2.aruco.DICT_4X4_250
)
charuco_board = cv2.aruco.CharucoBoard(
    (9, 6),
    0.04,
    0.02,
    aruco_dictionary,
)
board_image = charuco_board.generateImage((900, 600))

charuco_detector = cv2.aruco.CharucoDetector(charuco_board)
(
    charuco_corners,
    charuco_ids,
    marker_corners,
    marker_ids,
) = charuco_detector.detectBoard(board_image)

annotated_board = cv2.cvtColor(board_image, cv2.COLOR_GRAY2BGR)
if charuco_ids is not None:
    cv2.aruco.drawDetectedCornersCharuco(
        annotated_board,
        charuco_corners,
        charuco_ids,
    )

detected_count = 0 if charuco_ids is None else len(charuco_ids)
print(f"Esquinas ChArUco detectadas: {detected_count}")

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated_board, cv2.COLOR_BGR2RGB))
plt.title("Tablero ChArUco generado y esquinas detectadas")
plt.axis("off")
plt.show()


## 8. Ejercicios

**Ejercicio 1 — Calidad de las vistas.** Ordene las fotografías por error de reproyección. Inspeccione la mejor y la peor e indique qué factores visuales podrían explicar la diferencia.

**Resultado esperado:** desenfoque, poca cobertura del encuadre o ángulos extremos suelen empeorar la localización de esquinas.

**Ejercicio 2 — Cantidad de imágenes.** Repita la calibración con 3, 6 y todas las vistas válidas. Compare fx, fy y el RMS.

**Resultado esperado:** con pocas vistas los parámetros suelen ser menos estables; importa también que las poses sean variadas.

**Ejercicio 3 — Datos propios con ChArUco.** Imprima el tablero generado, capture entre 10 y 15 vistas y adapte la sección de detección. No use varias copias de una misma fotografía.

**Resultado esperado:** las vistas parciales todavía pueden aportar esquinas identificadas, una ventaja frente al tablero clásico.

**Checklist conceptual:** sabemos qué se mide, qué unidades dependen de square_size_cm, por qué hacen falta varias vistas y cómo verificar la calibración con reproyección.
